# Tesla — code-only Colab
Upload TESLA_COLAB_CODE_ONLY.zip. No media is bundled. This regenerates the existing base-v2 film, not the unimplemented premium re-cut. Frozen narration must be regenerated and match its recorded hash before rendering. Colab/GPU execution is untested; a mismatch stops rather than mixing incompatible timing. Full rendering is required.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile,os,sys,subprocess,json,hashlib
uploaded=files.upload()
archive=next(Path(n) for n in uploaded if n.endswith('.zip'))
root=Path('/content/tesla_code_only');root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
 for i in z.infolist():
  if not (root/i.filename).resolve().is_relative_to(root.resolve()) or ((i.external_attr>>16)&0o170000)==0o120000:raise ValueError('Unsafe archive')
 z.extractall(root)
project=root/'TESLA_COLAB_CODE_ONLY'
os.chdir(project)
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','venv','--python','3.13',str(project/'.colab-render-venv')],check=True)
subprocess.run([sys.executable,'-m','uv','pip','install','--python',str(project/'.colab-render-venv/bin/python'),'-r','explainer/colab/render-requirements.txt','kokoro-onnx==0.6.1'],check=True)
python=project/'.colab-render-venv/bin/python'



In [ ]:
import urllib.request
weights=project/'weights';weights.mkdir(exist_ok=True)
for name,expected in [('kokoro-v1.0.onnx','7d5df8ecf7d4b1878015a32686053fd0eebe2bc377234608764cc0ef3636a6c5'),('voices-v1.0.bin','bca610b8308e8d99f32e6fe4197e7ec01679264efed0cac9140fe9c29f1fbf7d')]:
 target=weights/name
 if not target.exists():urllib.request.urlretrieve('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/'+name,target)
 assert hashlib.sha256(target.read_bytes()).hexdigest()==expected
os.environ['KOKORO_MODEL']=str(weights/'kokoro-v1.0.onnx');os.environ['KOKORO_VOICES']=str(weights/'voices-v1.0.bin')
subprocess.run([str(python),'regenerate_voice.py'],check=True)
subprocess.run([str(python),'-m','explainer.projects.tesla-hw3.revision_colab','smoke'],check=True)
os.environ['EXPLAINER_V2_COLAB']='1'
subprocess.run([str(python),'-m','explainer.projects.tesla-hw3.produce_revision','--run'],check=True)


In [ ]:
from IPython.display import Video,display
movie=project/'explainer/projects/tesla-hw3/build/full-v2/tesla-planning-v2.mp4'
display(Video(str(movie),embed=True))
files.download(str(movie))
